<a href="https://colab.research.google.com/github/Goutham-26/DSA/blob/main/get_data_from_diff.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [110]:
import requests
import xml.etree.ElementTree as ET
import pandas as pd

In [111]:
csv = pd.read_csv("population.csv")
csv = csv.rename(columns={'country': 'Country'})
csv.shape

(25, 2)

In [112]:
excel = pd.read_excel("gdp.xlsx")
excel.shape

(25, 2)

In [113]:
json=pd.read_json("internet_users.json")
json = json.rename(columns={'country': 'Country'})
json.shape

(25, 2)

In [114]:

tree = ET.parse("literacy_rate.xml")
root = tree.getroot()
xml_list = []
for row in root:
    # Assuming 'country' and 'literacy_rate' are direct children of 'row' with lowercase tags
    country = row.find('country').text if row.find('country') is not None else None
    literacy_rate = row.find('literacy_rate').text if row.find('literacy_rate') is not None else None
    xml_list.append({"Country": country, "literacy_rate": literacy_rate})
xml = pd.DataFrame(xml_list)

xml.shape

(25, 2)

In [115]:
for df in [csv, excel, json, xml]:
    if "Country" in df.columns:
        df["Country"] = df["Country"].astype(str).str.strip().str.title()

In [116]:
csv["population"] = pd.to_numeric(
    csv["population"].astype(str).str.replace(",", ""), errors="coerce"
)
csv["population"] = csv["population"].fillna(
    csv["population"].median()
)

In [117]:
excel["GDP"] = pd.to_numeric(excel["GDP"], errors="coerce")
excel["GDP"] = excel["GDP"].fillna(0)

In [118]:
xml["literacy_rate"] = pd.to_numeric(
    xml["literacy_rate"], errors="coerce"
)

xml

,Country,literacy_rate
0,None,99.0
1,None,97.2
2,None,77.7
3,None,99.0
4,None,99.0
5,None,99.0
6,None,99.0
7,None,99.0
8,None,99.0
9,None,99.0


In [119]:
merged = csv.merge(excel, on="Country", how="outer")
merged = merged.merge(json, on="Country", how="outer")
merged = merged.merge(xml, on="Country", how="outer")
display(merged.head())

,Country,population,GDP,internet_users,literacy_rate
0,Argentina,4.519577e+07,640000.0,4.100000e+07,NaN
1,Australia,2.568704e+07,1880000.0,2.400000e+07,NaN
2,Brazil,2.125594e+08,2330000.0,1.810000e+08,NaN
3,Canada,3.800524e+07,2250000.0,3.600000e+07,NaN
4,China,1.411779e+09,19230000.0,1.080000e+09,NaN


In [120]:
merged["Internet Penetration Rate"] = merged.apply(
    lambda row: (
        (row["internet_users"] / row["population"]) * 100
        if row["population"] > 0
        else 0
    ),
    axis=1,
).round(2)

In [121]:
merged.columns.tolist()

['Country',
 'population',
 'GDP',
 'internet_users',
 'literacy_rate',
 'Internet Penetration Rate']

In [122]:
merged.head()

,Country,population,GDP,internet_users,literacy_rate,Internet Penetration Rate
0,Argentina,4.519577e+07,640000.0,4.100000e+07,NaN,90.72
1,Australia,2.568704e+07,1880000.0,2.400000e+07,NaN,93.43
2,Brazil,2.125594e+08,2330000.0,1.810000e+08,NaN,85.15
3,Canada,3.800524e+07,2250000.0,3.600000e+07,NaN,94.72
4,China,1.411779e+09,19230000.0,1.080000e+09,NaN,76.50


In [123]:
highest_internet_penetration = merged.sort_values(by='Internet Penetration Rate', ascending=False).head(5)
display(highest_internet_penetration[['Country', 'Internet Penetration Rate']])

,Country,Internet Penetration Rate
46,Switzerland,98.21
38,Norway,97.76
40,Saudi Arabia,97.66
45,Sweden,97.55
12,Netherlands,97.47


In [124]:
average_literacy_rate = merged['literacy_rate'].mean()
print(f"Average Literacy Rate: {average_literacy_rate:.2f}%")

Average Literacy Rate: 97.25%


In [125]:
analysis_columns = ['GDP', 'population', 'literacy_rate', 'Internet Penetration Rate']
correlation_matrix = merged[analysis_columns].dropna().corr()
display(correlation_matrix)

,GDP,population,literacy_rate,Internet Penetration Rate
GDP,NaN,NaN,NaN,NaN
population,NaN,NaN,NaN,NaN
literacy_rate,NaN,NaN,NaN,NaN
Internet Penetration Rate,NaN,NaN,NaN,NaN


In [126]:
import sqlite3
conn = sqlite3.connect('memory')

merged.to_sql('data', conn, if_exists='replace', index=False)

print("Merged DataFrame loaded into SQLite table 'data'.")

Merged DataFrame loaded into SQLite table 'data'.


In [127]:
query_a = """
SELECT Country, "Internet Penetration Rate" FROM data
ORDER BY "Internet Penetration Rate" DESC
LIMIT 5;
"""
highest_internet_penetration_sql = pd.read_sql_query(query_a, conn)
display(highest_internet_penetration_sql)

,Country,Internet Penetration Rate
0,Switzerland,98.21
1,Norway,97.76
2,Saudi Arabia,97.66
3,Sweden,97.55
4,Netherlands,97.47


In [128]:
query_b = """
SELECT AVG(literacy_rate) FROM data;
"""
average_literacy_rate_sql = pd.read_sql_query(query_b, conn)
print(f"Average Literacy Rate: {average_literacy_rate_sql.iloc[0,0]:.2f}%")

Average Literacy Rate: 97.25%


In [129]:
query_d = """
SELECT Country, GDP, population, (GDP * 1.0 / population) AS "GDP Per Capita" FROM data
WHERE population IS NOT NULL AND population > 0;
"""
gdp_per_capita_analysis = pd.read_sql_query(query_d, conn)
display(gdp_per_capita_analysis.head())

,Country,GDP,population,GDP Per Capita
0,Argentina,640000.0,4.519577e+07,0.014161
1,Australia,1880000.0,2.568704e+07,0.073189
2,Brazil,2330000.0,2.125594e+08,0.010962
3,Canada,2250000.0,3.800524e+07,0.059202
4,China,19230000.0,1.411779e+09,0.013621


In [130]:
query_e = """
SELECT Country, (GDP * 1.0 / population) AS "GDP Per Capita" FROM data
WHERE population IS NOT NULL AND population > 0 AND (GDP * 1.0 / population) > 10000
ORDER BY "GDP Per Capita" DESC;
"""
gdp_above_10k = pd.read_sql_query(query_e, conn)
display(gdp_above_10k)

,Country,GDP Per Capita


In [131]:
query_f = """
SELECT SUM(population) FROM data;
"""
total_population = pd.read_sql_query(query_f, conn)
print(f"Total population covered: {total_population.iloc[0,0]:,.0f}")

Total population covered: 4,720,039,725


In [132]:
query_g = """
SELECT Country, literacy_rate, "Internet Penetration Rate" FROM data
WHERE literacy_rate IS NOT NULL
ORDER BY literacy_rate ASC
LIMIT 5;
"""
lowest_literacy_rates = pd.read_sql_query(query_g, conn)
display(lowest_literacy_rates)

,Country,literacy_rate,Internet Penetration Rate
0,None,77.7,0.0
1,None,93.2,0.0
2,None,95.0,0.0
3,None,95.4,0.0
4,None,96.0,0.0


In [133]:
query_h = """
SELECT Country, GDP, population FROM data
ORDER BY GDP DESC
LIMIT 5;
"""
top_5_wealthiest = pd.read_sql_query(query_h, conn)
display(top_5_wealthiest)

,Country,GDP,population
0,United States,30500000.0,3.310027e+08
1,China,19230000.0,1.411779e+09
2,Germany,4744000.0,8.324052e+07
3,India,4187000.0,1.380004e+09
4,Japan,4180000.0,1.258360e+08


In [134]:
query_i = """
SELECT Country, "Internet Penetration Rate" FROM data
WHERE "Internet Penetration Rate" > 70
ORDER BY "Internet Penetration Rate" DESC;
"""
internet_over_70_percent = pd.read_sql_query(query_i, conn)
display(internet_over_70_percent)

,Country,Internet Penetration Rate
0,Switzerland,98.21
1,Norway,97.76
2,Saudi Arabia,97.66
3,Sweden,97.55
4,Netherlands,97.47
5,United Kingdom,97.22
6,South Korea,96.56
7,Singapore,95.72
8,France,94.98
9,Germany,94.91


In [135]:
query_j = """
SELECT AVG(GDP * 1.0 / population) FROM data
WHERE "Internet Penetration Rate" > 50 AND population IS NOT NULL AND population > 0;
"""
average_gdp_per_capita_high_internet = pd.read_sql_query(query_j, conn)
print(f"Average GDP Per Capita for countries with >50% Internet Penetration: ${average_gdp_per_capita_high_internet.iloc[0,0]:,.2f}")

Average GDP Per Capita for countries with >50% Internet Penetration: $0.04


In [136]:
query_k = """
SELECT COUNT(Country) AS NumCountries, AVG("Internet Penetration Rate") AS AverageInternetPenetration
FROM data
WHERE literacy_rate > 90;
"""
high_literacy_insights = pd.read_sql_query(query_k, conn)
display(high_literacy_insights)

,NumCountries,AverageInternetPenetration
0,24,0.0
